In [4]:
import pandas as pd

raw = pd.read_excel("../data/raw/Online Retail.xlsx",
                    dtype={"InvoiceNo": str, "StockCode": str})
df = raw.copy()
log = [("raw data", len(df))]
print(df.shape)

(541909, 8)


In [2]:
neg = df[df["Quantity"] < 0]
non_cancel_neg = neg[~neg["InvoiceNo"].str.startswith("C")]
print(len(non_cancel_neg))
print((non_cancel_neg["UnitPrice"] == 0).mean())
print(non_cancel_neg[["Description", "Quantity", "UnitPrice"]].head(10))

1336
1.0
     Description  Quantity  UnitPrice
2406         NaN       -10        0.0
4347         NaN       -38        0.0
7188         NaN       -20        0.0
7189         NaN       -20        0.0
7190         NaN        -6        0.0
7192         NaN       -22        0.0
7193         NaN        -6        0.0
7195         NaN        -2        0.0
7196         NaN       -30        0.0
7197         NaN       -70        0.0


In [5]:

is_cancel = df["InvoiceNo"].str.startswith("C")
returns = df[is_cancel].copy()
df = df[~is_cancel]
log.append(("removed cancelled invoices", len(df)))


df = df[(df["Quantity"] > 0) & (df["UnitPrice"] > 0)]
log.append(("removed non-positive qty/price", len(df)))

df = df[df["StockCode"].str.match(r"^\d{5}[A-Za-z]*$")]
log.append(("removed non-product codes", len(df)))


df = df.dropna(subset=["Description"])
log.append(("removed missing description", len(df)))


df = df.drop_duplicates()
log.append(("removed exact duplicates", len(df)))

In [6]:
df = df[(df["InvoiceDate"] >= "2010-12-06") & (df["InvoiceDate"] < "2011-12-05")]
log.append(("trimmed to full weeks (6 Dec 2010 to 4 Dec 2011)", len(df)))

In [7]:
df["Revenue"] = df["Quantity"] * df["UnitPrice"]

print(df["Quantity"].describe())
print(df.nlargest(10, "Quantity")[["StockCode", "Description", "Quantity", "UnitPrice", "InvoiceDate"]])

count    495525.000000
mean         10.546788
std         111.985284
min           1.000000
25%           1.000000
50%           4.000000
75%          12.000000
max       74215.000000
Name: Quantity, dtype: float64
       StockCode                          Description  Quantity  UnitPrice  \
61619      23166       MEDIUM CERAMIC TOP STORAGE JAR     74215       1.04   
421632     84077    WORLD WAR 2 GLIDERS ASSTD DESIGNS      4800       0.21   
206121     22197                 SMALL POPCORN HOLDER      4300       0.72   
97432      22053                EMPIRE DESIGN ROSETTE      3906       0.82   
270885     18007  ESSENTIAL BALM 3.5g TIN IN ENVELOPE      3186       0.06   
52711      21108   FAIRY CAKE FLANNEL ASSORTED COLOUR      3114       2.10   
160546     21108   FAIRY CAKE FLANNEL ASSORTED COLOUR      3114       2.10   
433788     16014          SMALL CHINESE STYLE SCISSOR      3000       0.32   
291249     84879        ASSORTED COLOUR BIRD ORNAMENT      2880       1.45   
20114

In [8]:
log_df = pd.DataFrame(log, columns=["step", "rows_remaining"])
log_df["rows_removed"] = -log_df["rows_remaining"].diff().fillna(0).astype(int)
print(log_df)
print("Products left:", df["StockCode"].nunique())

                                               step  rows_remaining  \
0                                          raw data          541909   
1                        removed cancelled invoices          532621   
2                    removed non-positive qty/price          530104   
3                         removed non-product codes          527725   
4                       removed missing description          527725   
5                          removed exact duplicates          522504   
6  trimmed to full weeks (6 Dec 2010 to 4 Dec 2011)          495525   

   rows_removed  
0             0  
1          9288  
2          2517  
3          2379  
4             0  
5          5221  
6         26979  
Products left: 3882


In [10]:
df.to_parquet("../data/processed/clean_sales.parquet", index=False)
returns.to_parquet("../data/processed/returns.parquet", index=False)

## Cleaning decisions

| Step | Decision | Reason |
|---|---|---|
| Cancelled invoices | Moved to a separate `returns` table | Demand means what customers wanted to buy. Returns happen later and would create fake negative demand. |
| Zero/negative price or quantity | Removed (1,336 stock adjustments with no description) | These are write-offs and adjustments, not sales. |
| Non-product codes | Removed (postage, fees, manual entries) | They are not products and cannot be forecast or stocked. |
| Exact duplicates | Removed | Repeated rows would inflate demand. |
| Date range | Trimmed to 52 full weeks (6 Dec 2010 to 4 Dec 2011) | The first and last weeks are partial and would look like false demand drops. |
| Missing CustomerID | Kept (about 25% of rows) | Demand does not depend on who bought, and dropping them would remove a quarter of sales. |

**Result:** 541,909 raw rows became 495525 clean rows covering 3882 products.

**Limitation:** only about one year of data, so a year-lag seasonal baseline is not possible.


In [11]:
print(len(df), df["StockCode"].nunique())

495525 3882
